# 0. RLHF Overview — 큰 그림 잡기

이 시리즈는 [Nathan Lambert의 *RLHF 책* (arXiv:2504.12501)](https://arxiv.org/abs/2504.12501)에 나오는 **클래식 RLHF 풀 루프**를 4개 노트북에 걸쳐 from-scratch PyTorch로 구현하면서 깊이 이해하는 것이 목적입니다.

## 4-노트북 시리즈 로드맵

| 노트북 | 단계 | 핵심 알고리즘 | 책의 챕터 |
|--------|------|---------------|-----------|
| **0_Overview** (이 노트북) | 배경 | RL primer, KL, 파이프라인 다이어그램 | Ch 1~4 |
| 1_SFT | Supervised Fine-Tuning | Cross-entropy with response masking | Ch 5 |
| 2_RewardModel | 선호 학습 | Bradley-Terry + sigmoid loss | Ch 6 |
| 3_PPO | 강화학습 | PPO clipped objective + KL penalty | Ch 8 |

> 실행되지는 않습니다 (Kybalion-1B 모델 로드 + GPU 필요). 코드를 **읽으면서 RLHF 구조를 완벽히 이해하는 것**이 목적입니다.

## 1. 왜 RLHF인가?

사전학습된 LLM은 인터넷 텍스트에서 다음 토큰을 예측하도록 학습됩니다. 결과:

- ✅ 문법 / 사실 지식은 잘 습득
- ❌ "사용자가 원하는 답을 정중하게 제공" 같은 *행동 양식*은 학습 데이터에 명시되지 않음
- ❌ 위험한 요청에 거부, 모르면 모른다고 말하기 등 *가치 정렬*도 부재

**RLHF는 이 gap을 메우는 3단계 절차**:

```
사전학습 모델  →  SFT  →  Reward Model  →  PPO (RLHF)  →  정렬된 모델
   (지식)      (포맷 학습)   (선호 학습)    (선호 최대화)
```

핵심 가정: **인간의 선호 데이터로부터 학습된 reward 함수를 RL의 보상 신호로 사용하면, 모델이 인간 선호도가 높은 응답을 생성하게 된다.**

## 2. 강화학습 (RL) 기초 복습

LLM에 적용하기 전에 RL의 5요소를 정리:

| 기호 | 이름 | LLM 맥락에서 의미 |
|------|------|------------------|
| $s$ | 상태 (state) | 지금까지 생성된 토큰 시퀀스 |
| $a$ | 행동 (action) | 다음에 생성할 토큰 |
| $\pi_\theta(a \mid s)$ | 정책 (policy) | LLM 자체 — 토큰에 대한 확률분포 |
| $r(s, a)$ | 보상 (reward) | RLHF에선 응답 끝에 한 번 (sequence-level) |
| $\gamma$ | 할인 (discount) | 보통 1.0 (시퀀스가 짧으니까) |

### 목표 함수

$$
J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\bigl[R(\tau)\bigr]
\quad \text{where} \quad \tau = (s_0, a_0, s_1, a_1, \ldots), \quad R(\tau) = \sum_t r(s_t, a_t)
$$

말하자면: **정책 π_θ가 생성하는 응답들의 기대 누적 보상을 최대화**한다.

LLM에서 τ는 응답 시퀀스이고, $R(\tau)$는 그 응답에 대한 인간(또는 reward model)의 만족도 점수다.

## 3. 정책 경사 (Policy Gradient) — RL의 출발점

위 목표 $J(\theta)$를 어떻게 최대화하나? **경사 상승법** — 단, $J$는 stochastic이라 직접 미분 안 됨.

**Policy Gradient Theorem** (Williams 1992):

$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}
\Bigl[\sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot R(\tau)\Bigr]
$$

### 한국어 해석

각 step에서 "그 토큰을 선택할 확률에 log를 취하고, 거기에 전체 보상을 곱해서 합산한다." 보상이 높았던 trajectory의 토큰들의 확률은 ↑, 낮았던 trajectory의 토큰 확률은 ↓.

### 도전 과제

- **분산이 큼**: $R(\tau)$가 sequence-level이라 noisy
- **샘플 비효율**: 매번 새 rollout 필요
- **과도한 업데이트 위험**: 새 정책이 너무 멀어지면 안정성 ↓

→ PPO는 이 세 문제를 해결하는 방식. (3번 노트북에서 자세히)

## 4. KL 발산 — RLHF의 안전벨트

**문제**: RL이 reward만 최대화하다 보면 SFT 모델에서 너무 멀어져서 횡설수설하게 됨 ("reward hacking").

**해결**: 학습 중인 정책 π_θ가 reference (SFT 모델) π_ref와 너무 달라지지 못하게 KL 발산으로 제약.

$$
D_{\mathrm{KL}}(\pi_\theta \,\|\, \pi_{\mathrm{ref}}) =
\mathbb{E}_{a \sim \pi_\theta}\Bigl[\log \frac{\pi_\theta(a \mid s)}{\pi_{\mathrm{ref}}(a \mid s)}\Bigr]
$$

### 직관

$D_\mathrm{KL} = 0$이면 두 분포가 동일. 클수록 멀어짐. **양방향이 아닌 점**에 주의 — $D_\mathrm{KL}(P\|Q) \ne D_\mathrm{KL}(Q\|P)$.

### RLHF에서 어떻게 쓰이나?

전체 목표는 *수정된* J:

$$
J_{\mathrm{RLHF}}(\theta) =
\mathbb{E}_{x \sim \mathcal{D},\; y \sim \pi_\theta(\cdot \mid x)}
\bigl[ r_\phi(x, y) \bigr] -
\beta \cdot D_{\mathrm{KL}}\bigl(\pi_\theta(\cdot \mid x) \,\|\, \pi_{\mathrm{ref}}(\cdot \mid x)\bigr)
$$

- $r_\phi$: reward model의 점수
- $\beta$: KL 패널티 강도 (보통 0.01~0.2)
- $\pi_\mathrm{ref}$: SFT 모델 (학습 중에 변화 없음)

**β가 작으면**: reward를 빡세게 추구. 모델이 이상해질 수 있음.
**β가 크면**: SFT에서 거의 못 떠남. 정렬 효과 약함.

## 5. RLHF 풀 파이프라인 다이어그램

```
┌─────────────────────────────────────────────────────────────┐
│  Stage 1: Supervised Fine-Tuning (SFT)                       │
│                                                                │
│   Pre-trained LM  +  (prompt, demonstration) data            │
│         │                                                      │
│         ↓  cross-entropy on response tokens only             │
│         │                                                      │
│      π_SFT  (이게 이후 모든 단계의 시작점)                    │
└─────────────────────────────────────────────────────────────┘
                              │
              ┌───────────────┼───────────────┐
              ↓               ↓               ↓
┌─────────────────────────────────────────────────────────────┐
│  Stage 2: Reward Modeling                                     │
│                                                                │
│   π_SFT 복사 + scalar head  +  preference data (A > B)      │
│         │                                                      │
│         ↓  Bradley-Terry loss                                │
│         │                                                      │
│       r_φ  (응답에 점수를 매기는 함수)                       │
└─────────────────────────────────────────────────────────────┘
                              │
                              ↓
┌─────────────────────────────────────────────────────────────┐
│  Stage 3: PPO with KL penalty                                 │
│                                                                │
│   π_θ (= π_SFT 복사, 학습 대상)                              │
│   π_ref (= π_SFT 복사, 고정)                                 │
│   r_φ (Reward model, 고정)                                   │
│   V_ψ (Value head, 학습 대상)                                │
│                                                                │
│   Loop:                                                        │
│     1. π_θ로 응답 샘플링                                     │
│     2. r_φ로 reward 계산                                     │
│     3. KL(π_θ || π_ref) 계산                                 │
│     4. Advantage = R - V  계산                               │
│     5. PPO clipped objective로 π_θ 업데이트                 │
│     6. Value 회귀로 V_ψ 업데이트                             │
│                                                                │
│       π_RLHF  ← 최종 정렬된 모델                              │
└─────────────────────────────────────────────────────────────┘
```

### 메모리 관점

PPO 단계에선 동시에 4개 모델이 메모리에 떠 있음:
1. **π_θ** (policy, 학습)
2. **π_ref** (reference, 고정, KL 계산)
3. **r_φ** (reward, 고정)
4. **V_ψ** (value/critic, 학습)

→ 7B 모델이면 4×14GB = ~56GB. **이게 DPO가 인기 있는 이유** (RM과 critic 둘 다 제거됨).

## 6. 우리가 쓸 표기법 정리

| 기호 | 의미 |
|------|------|
| $x$ | 프롬프트 (e.g. "양자얽힘을 설명해줘") |
| $y$ | 응답 토큰 시퀀스 $y = (y_1, \ldots, y_T)$ |
| $\pi_\theta(y \mid x)$ | 정책: $\prod_{t} \pi_\theta(y_t \mid x, y_{<t})$ |
| $\pi_\mathrm{SFT}$ | SFT 단계 종료 시점의 정책 |
| $\pi_\mathrm{ref}$ | 참조 정책 (보통 = $\pi_\mathrm{SFT}$, 고정) |
| $r_\phi(x, y)$ | 학습된 reward model의 점수 (scalar) |
| $\beta$ | KL 패널티 강도 |
| $A_t$ | advantage at step $t$ |
| $V_\psi(s_t)$ | value function (critic) |
| $\mathcal{D}$ | 학습 데이터 (preference dataset 또는 prompt set) |

## 7. 실습 환경 — Kybalion-1B narrative

이 시리즈는 [devwoo/Kybalion-1B](https://huggingface.co/devwoo/Kybalion-1B) (Llama 3.2 1B 기반, CPT+SFT 완료)을 가상의 시작점으로 가정합니다.

| 단계 | Kybalion 입장에서 의미 |
|------|----------------------|
| 사전학습 | Meta가 Llama 3.2 1B 사전학습 |
| (도메인 적응) | 사용자가 CPT 3.5B 토큰 |
| **SFT** | 사용자가 LoRA로 instruction tuning 완료 ✅ |
| **Reward Model** | 본 시리즈 #2에서 다룸 |
| **PPO** | 본 시리즈 #3에서 다룸 |

실제로는 이미 SFT까지 완료된 모델이지만, 본 노트북에선 처음부터 RLHF 풀 루프를 다시 처음부터 보여줍니다.

## 8. 가벼운 PyTorch sanity check

이 시리즈에서 쓸 환경 / import를 점검.

In [ ]:
# 본 시리즈에서 사용할 import. 실제로 안 돌리더라도 코드를 읽으면서 따라가실 때
# 이 셀의 import 목록만 봐도 어떤 stack을 쓰는지 한눈에 보입니다.
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# HuggingFace stack (모델 로드 / 토크나이저)
from transformers import AutoTokenizer, AutoModelForCausalLM

# 단순 RL 환경 (필요 시)
import numpy as np
from dataclasses import dataclass
from typing import Optional

# 시연 출력용
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 9. KL 발산 직접 계산해보기 (이해 확인용)

두 카테고리컬 분포의 KL을 직접 계산. RLHF에선 토큰 vocab에 대한 KL을 매 step 계산해야 합니다.

In [ ]:
# 예시: 작은 vocab (5개 토큰)에서 두 분포의 KL 계산
#   - π_theta : 학습 중인 정책 (조금 다름)
#   - π_ref   : 참조 정책 (SFT 종료 시)

# logits로 시작 (실제 LLM 출력 형태)
logits_theta = torch.tensor([2.0, 1.0, 0.5, -1.0, -0.5])
logits_ref   = torch.tensor([1.8, 1.2, 0.3, -0.8, -0.4])

# softmax로 확률 분포 변환
p_theta = F.softmax(logits_theta, dim=-1)
p_ref   = F.softmax(logits_ref, dim=-1)
print("π_theta:", p_theta.numpy().round(3))
print("π_ref  :", p_ref.numpy().round(3))

# ---- 정의식대로 KL 계산 ----
#   D_KL(π_theta || π_ref) = Σ π_theta(a) * log(π_theta(a) / π_ref(a))
kl_manual = (p_theta * (p_theta.log() - p_ref.log())).sum()
print(f"\n수동 KL: {kl_manual.item():.4f}")

# ---- PyTorch built-in으로 검증 ----
#   F.kl_div는 log P_target과 P_input을 받음. 인자 순서 헷갈리니 주의.
kl_torch = F.kl_div(
    input=p_ref.log(),    # log Q (참조 분포의 log-prob)
    target=p_theta,        # P  (학습 분포)
    reduction="sum",
)
print(f"PyTorch KL: {kl_torch.item():.4f}  # 동일해야 함")

## 10. RLHF의 KL 추정 트릭

실제 RLHF 학습에선 분포 전체를 매번 계산하지 않고, **샘플 기반 추정자**를 씁니다.

가장 흔한 estimator (book §8.1):

$$
\widehat{D}_{\mathrm{KL}}(\pi_\theta \,\|\, \pi_{\mathrm{ref}}) =
\log \pi_\theta(y \mid x) - \log \pi_{\mathrm{ref}}(y \mid x)
$$

즉 **한 샘플 $y$를 뽑아서, 그 샘플의 log-likelihood 차이**만 본다. 단점: 분산이 큼. PPO는 이를 보완하기 위해 더 정교한 추정자를 씀 (e.g. *k1*, *k3* in Schulman's blog).

In [ ]:
# 위 estimator를 코드로 작성
# 입력: per-token log probs.
#  log_p_theta: (T,) — 학습 중 정책의 토큰별 log P
#  log_p_ref:   (T,) — 참조 정책의 토큰별 log P

def kl_estimator_k1(log_p_theta: torch.Tensor, log_p_ref: torch.Tensor) -> torch.Tensor:
    """K1 estimator (Schulman, 2020) — 단순 차이.

    D_KL ≈ E[log π_θ(y) - log π_ref(y)] when y ~ π_θ
    한 sequence에 대한 추정. token-level이면 sum, sequence-level이면 그대로.
    """
    return (log_p_theta - log_p_ref).sum(dim=-1)


def kl_estimator_k3(log_p_theta: torch.Tensor, log_p_ref: torch.Tensor) -> torch.Tensor:
    """K3 estimator — unbiased, lower variance (Schulman, 2020).

    D_KL ≈ E[ratio - 1 - log(ratio)]
         = E[exp(log_ratio) - 1 - log_ratio]
    """
    log_ratio = log_p_theta - log_p_ref
    return (log_ratio.exp() - 1.0 - log_ratio).sum(dim=-1)


# 가상의 토큰 5개에 대해 log-prob 시뮬레이션
log_p_theta = torch.tensor([-0.10, -0.20, -0.05, -0.30, -0.15])
log_p_ref   = torch.tensor([-0.12, -0.18, -0.06, -0.28, -0.16])

print(f"K1 KL estimate: {kl_estimator_k1(log_p_theta, log_p_ref).item():.5f}")
print(f"K3 KL estimate: {kl_estimator_k3(log_p_theta, log_p_ref).item():.5f}")
# K3는 항상 ≥ 0이고 unbiased. K1은 음수도 나올 수 있음 (분산 큼).

## 11. 다음 노트북 — `1_SFT.ipynb`

다음 노트북에서는 **SFT (Supervised Fine-Tuning)** 의 수학과 구현을 다룹니다.

- Cross-entropy loss를 응답 토큰에만 적용하는 마스킹 트릭
- Llama 3.2 chat template과 SFT loss의 상호작용
- Kybalion이 이미 거친 단계를 처음부터 작성

이 단계의 산출물 $π_\mathrm{SFT}$가 이후 **모든** 단계의 초기값이 됩니다.